[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HumbertoDiego/AjustamentoBasicoIME/blob/main/UD_III.ipynb)

# Ajustamento Básico - UD III
**Maj Diego - 2° Semestre / 2026**

UD III

1. Propagação das variâncias para um modelo matemático linear. 

- Relacionar a propagação das variâncias aos modelos matemáticos lineares. 

2. Propagação das variâncias para um modelo matemático não-linear. 

- Relacionar a propagação das variâncias aos modelos matemáticos não lineares. 

3. Aplicações práticas da lei de propagação das variâncias.

- Exemplificar casos lineares e não lineares com aplicações da Eng. Cartográfica.

Referência:

Ghilani, C. D. (2017). *Adjustment computations: Spatial data analysis* (6th ed.). Wiley. $\rightarrow$ **Cap 6, 7, 8 e 9**

## O Problema

Sabendo a precisão das observações, como obter a precisão do modelo?

## UD III

### 1. Propagação das variâncias para um modelo matemático linear.

Uma vez que todas as quantidades diretamente observadas contêm erros, quaisquer valores calculados a partir delas também conterão erros. Essa intrusão, ou propagação é um dos mais importantes discutidos nesta disciplina. 

A seguir, assume-se que todos os erros sistemáticos e equívocos foram eliminados das observações diretas, de modo que apenas os erros aleatórios permaneçam.

#### O que é variância?

Resposta rápida: é o quadrado desvio padrão, ou seja, $\sigma^2$. 😑😑😑😑

A variância, assim como o desvio-padrão, é uma medida de **dispersão em relação à média**. Em termos algébricos:

$$
 \sigma_Z^2 = \frac{1}{gl} \sum_{i=1}^{m} (Z^i - \mu_{Z})^2 = E(( Z − \mu_Z)^2)
$$

onde $Z$ é uma variável aleatória $m$-dimensional (medida $m$ vezes), $\mu_{Z}$ é sua média, $Z^i$ é um de seus componentes (uma medição), $gl$ é o grau de liberdade do sistema (em se tratando de amostra, $gl=m-1$, em se tratando de população, $gl=m$, em se tratando de um sistema de $m$ equações e $u$ parâmetros, $gl=m-u$) e, para fins de demonstrar a propagação dos erros, vamos utilizar a notação de operador linear de esperança $E$.

> Nota: perceba dentro do operador esperança que, a princípio, $Z$ vetorial e $\mu_Z$ ecalar não permitem a subtração. Está implícito o conceito de *broadcast* da algebra vetorial e de *arrays* em computação, onde a grandeza escalar tem sua dimensionalidade aumentada, repetida tantas vezes quantas forem necessárias para permitir a operação elemento a elemento.

Sob a ótica do ajustamento e em muitas aplicações, a variância não é calculada para
cada observação (isto seria caro!), mas retiradas das especificações técnicas do equipamento ou de posteriores
calibrações. 

<img src="media/imgs/img4.png" width=400>

#### O que é covariância?

A covariância mensura a dependência linear entre 2 variáveis unidimensionais.

- Para o caso de variáveis independentes, a covariância entre elas é nula;
- Sejam $Z'$ e $Z''$ duas variáveis com alguma relação de dependência entre si. A covariância entre elas é dada por:

$$
\sigma_{Z',Z''} = E[(Z' - \mu_{Z'})(Z'' - \mu_{Z''})]
$$

**Exercício 01:** Uma nuvem de pontos que representa um plano:

In [ ]:
import numpy as np
import open3d as o3d

# Equação do plano: ax + by + cz + d = 0
a, b, c, d = 1, 2, -1, 5  # Exemplo de plano qualquer
print(f"Gerando o plano {a}x + {b}y + {c}z + {d} = 0")

# Gerar pontos na nuvem
num_points = 1000
x = np.random.uniform(-10, 10, num_points)
y = np.random.uniform(-10, 10, num_points)
z = (-a * x - b * y - d) / c

# Criar array de pontos
points = np.column_stack((x, y, z))
print("XYZ:\n",points)

# Criar nuvem de pontos com Open3D
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)

# Visualizar a nuvem de pontos
o3d.visualization.draw_plotly([pcd], width=400, height=300)

Gerando o plano 1x + 2y + -1z + 5 = 0
XYZ:
 [[-8.38484173  2.82718525  2.26952877]
 [-8.40826063 -0.75403761 -4.91633585]
 [ 8.53013037  8.30737274 30.14487585]
 ...
 [-5.0321856  -2.87122231 -5.77463023]
 [ 9.46271465 -0.77360559 12.91550347]
 [-5.28721966 -1.46591983 -3.21905933]]


In [32]:
# Covariâncias
cov_xy = np.cov(points[:, 0], points[:, 1])[0, 1]
cov_xz = np.cov(points[:, 0], points[:, 2])[0, 1]
cov_yz = np.cov(points[:, 1], points[:, 2])[0, 1]
print("Covariância de x com y:", cov_xy)
print("Covariância de x com z:", cov_xz)
print("Covariância de y com z:", cov_yz)

Covariância de x com y: 1.577004305322919
Covariância de x com z: 36.54632436316327
Covariância de y com z: 68.89745070242058


Quais as variáveis independentes? 

Qual o grau de dependênica entre cada variável, comparando duas a duas?

**Exercício 02:** Uma nuvem de pontos que representa uma reconstrução 3D:

In [24]:
import os
import wget
import open3d as o3d
import numpy as np

sample = "data/bunny.pcd"

if os.path.exists(sample):
    print("O arquivo 'bunny.pcd' já existe. Pulando download.")
else:
    url = 'https://raw.githubusercontent.com/PointCloudLibrary/pcl/master/test/bunny.pcd'
    print(f"Baixando o arquivo 'bunny.pcd' de {url}...")
    wget.download(url, sample)
    print("Download concluído.")

pcd = o3d.io.read_point_cloud(sample)
xyz = np.asarray(pcd.points)
print("XYZ:\n",xyz)

eixos = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.06, origin=[0,0,0])
o3d.visualization.draw_plotly([eixos,pcd], width=400, height=300)

O arquivo 'bunny.pcd' já existe. Pulando download.
XYZ:
 [[ 0.0054216  0.11349    0.040749 ]
 [-0.0017447  0.11425    0.041273 ]
 [-0.010661   0.11338    0.040916 ]
 ...
 [-0.064992   0.17802   -0.054645 ]
 [-0.069935   0.17983   -0.051988 ]
 [-0.07793    0.17516   -0.0444   ]]


In [34]:
# Covariâncias
cov_xy = np.cov(xyz[:, 0], xyz[:, 1])[0, 1]
cov_xz = np.cov(xyz[:, 0], xyz[:, 2])[0, 1]
cov_yz = np.cov(xyz[:, 1], xyz[:, 2])[0, 1]
print("Covariância de x com y:", cov_xy)
print("Covariância de x com z:", cov_xz)
print("Covariância de y com z:", cov_yz)

Covariância de x com y: -0.0005930386397008904
Covariância de x com z: 0.00017801328678129232
Covariância de y com z: -0.0005568275720287393


Quais as variáveis independentes? 

Qual o grau de dependênica entre cada variável, comparando duas a duas?

#### Matriz Variância-Covariância (MVC ou $\Sigma$) das observações

É o caso geral para se representar as variâncias e covariâncias de uma distribuição multivariada $Z$, ou seja, função de outras variáveis. Em termos algébricos: $Z = f(Z_1,...,Z_n)$ onde cada $Z_k$ representa uma variável aleatória $m$-dimensional (medida $m$ vezes independentemente) 

- Elementos na posição $(i,j)$ da matriz representam covariâncias $\sigma_{Z_i,Z_j}$
- Com isso, a diagonal principal é composta pelas variâncias $\sigma^2_{Z_i}$ (notar que a VAR é um caso particular da COV com $i=j$)

$$
\mathrm{MVC}(\mathbf{Z})
= \begin{bmatrix}
\sigma_{Z_1,Z_1} & \sigma_{Z_1,Z_2} & \cdots & \sigma_{Z_1,Z_n} \\
\sigma_{Z_2,Z_1} & \sigma_{Z_2,Z_2} & \cdots & \sigma_{Z_2,Z_n} \\
\vdots & \vdots & \ddots & \vdots \\
\sigma_{Z_n,Z_1} & \sigma_{Z_n,Z_2} & \cdots & \sigma_{Z_n,Z_n}
\end{bmatrix}
$$

**Exercício 03:** Dada uma simples função:

$$
z = ax + by
$$

onde $x$ e $y$ são variáveis aleatórias observadas independentemente com desvios padrão $\sigma_x$ e $\sigma_y$ e $a$ and $b$ são constantes. 

$z$ assume o papel de uma distribuição bivariada, a qual desejamos encontrar seu desvio padrão $\sigma_z$.

#### Lei Geral de Propagação de Covariâncias


### 2. O método dos mínimos quadrados (MMQ)